# CAMUS — DRL: LV\_endo (Class 1)

**Set `AGENT_NAME` in Cell 1 to pick the algorithm. Everything else is automatic.**

| Variable | Options |
|---|---|
| `AGENT_NAME` | `'DQN'` · `'DuelingDDQN'` · `'DDPG'` |

Reward shaping for this class: **`dice_hd_composite`** (α=0.6 Dice / β=0.4 HD95, hd\_norm=12 px).  
Rationale: baseline Dice ≈ 0.938 is saturated — HD95 adds boundary-precision signal.

> Supersedes `06_camus_drl.ipynb` for LV\_endo. Run §3 (dry-run) first, then §4 (full).

## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'✓ Package at: {PKG_ROOT}')

## 1 · Configure — set algorithm here

In [ ]:
# ══════════════════════════════════════════════════════════════════════
AGENT_NAME = 'DQN'   # DQN | DuelingDDQN | DDPG  (DQN, DuelingDDQN = 14-action local mask refinement; DDPG = continuous baseline)
# ══════════════════════════════════════════════════════════════════════

from iteris.config import load_drl_class_config, resolve_agent_config, load_config
from iteris.utils  import get_device

cfg_full     = load_drl_class_config(str(PKG_ROOT / 'configs' / 'camus_drl_c1.yaml'))
cfg          = resolve_agent_config(cfg_full, AGENT_NAME)
baseline_cfg = load_config(str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

cfg['checkpoint_dir'] = '/kaggle/working'

# Auto-detect CAMUS root and U-Net checkpoint
camus_roots = [p for p in Path('/kaggle/input').rglob('CAMUS') if p.is_dir()]
if camus_roots:
    baseline_cfg['data_root'] = str(camus_roots[0])
ckpt_cands = [p for p in Path('/kaggle/input').rglob('camus_best.pt')]
if ckpt_cands:
    cfg['baseline_checkpoint'] = str(ckpt_cands[0])

print(f'Agent           : {cfg["agent_type"]}')
print(f'Target class    : {cfg["target_class"]} ({cfg["class_name"]})')
print(f'Reward mode     : {cfg["reward_mode"]}  '
      f'(α={cfg.get("reward_alpha","-")} β={cfg.get("reward_beta","-")} '
      f'hd_norm={cfg.get("hd_norm","-")})')
print(f'CAMUS data root : {baseline_cfg["data_root"]}')
print(f'Baseline ckpt   : {cfg["baseline_checkpoint"]}')
print(f'Train steps     : {cfg["train_steps"]}')
print(f'Device          : {get_device()}')

## 2 · Warm-start — pre-compute U-Net init masks
~5 min for CAMUS. Caches U-Net predictions so the DRL loop never re-runs the backbone.

In [ ]:
from iteris.warm_start import precompute_init_masks
import numpy as np

train_samples, val_samples, test_samples = precompute_init_masks(
    baseline_cfg        = baseline_cfg,
    baseline_checkpoint = cfg['baseline_checkpoint'],
    target_class        = cfg['target_class'],
    min_area_fraction   = cfg.get('min_area_fraction', 0.01),
)
print(f'\nSamples — train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}')

init_dices = [float((s['init_mask'] & s['gt_mask']).sum() * 2 /
                    (s['init_mask'].sum() + s['gt_mask'].sum() + 1e-6))
              for s in val_samples]
print(f'U-Net init Dice on val ({cfg["class_name"]}): '
      f'mean {np.mean(init_dices):.4f}  median {np.median(init_dices):.4f}')

## 3 · OPTIONAL — Dry-run sanity check (~3 min)
Self-contained `deepcopy(cfg)` — does NOT mutate `cfg`, `train_samples`, or `val_samples`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
RUN_DRY_RUN = False   # Set True for a quick ~2-3 min sanity check before full training
# ══════════════════════════════════════════════════════════════════════

if RUN_DRY_RUN:
    import copy
    from iteris.drl_training import run_drl_training

    # Override values are paradigm-agnostic: for boundary tracing each "step"
    # is one batched iteration over num_envs (16) envs; for DDPG it is one env
    # step. buffer_size/prefill are sized to comfortably cover both.
    _dry = copy.deepcopy(cfg)
    _dry.update(dict(train_steps=600, prefill_steps=500, buffer_size=3000,
                     eval_every=200, epsilon_decay_steps=400, batch_size=32))
    _dry_result = run_drl_training(_dry, train_samples[:30], val_samples[:10])
    print(f'\n✓ Dry run passed. Best val score (meaningless at 600 steps): {_dry_result["best_dice"]:.4f}')
    print(f'  cfg["train_steps"] still = {cfg["train_steps"]}  (unchanged)')
    print('  → Safe to run §4 below for the real training run.')
else:
    print('Dry run skipped (RUN_DRY_RUN = False). Proceed directly to §4.')


## 4 · Full training
DQN / DuelingDDQN (boundary tracing): ~1.5–2 hr · DDPG (continuous baseline): ~5–6 hr on T4.
Run this **or** §3 — not both.


In [ ]:
from iteris.drl_training import run_drl_training

result    = run_drl_training(cfg, train_samples, val_samples)
agent     = result['agent']
history   = result['history']     # pandas DataFrame
best_dice = result['best_dice']

print(f"\n✓ Training complete — {cfg['agent_type']} on {cfg['class_name']}")
print(f"  Best val final-Dice : {best_dice:.4f}")
print(f"  Checkpoint          : {result['checkpoint']}")


## 5 · Visualisation setup
Defines `replay_episode()` and `ENV_KW`. Used by all cells below.

In [ ]:
from iteris.refinement_viz import (
    refinement_env_kwargs, build_replays, plot_comparison,
    plot_playback, plot_behaviour, evaluate_testset)

ENV_KW  = refinement_env_kwargs(cfg)
N_VIZ   = 8
replays = build_replays(agent, val_samples, ENV_KW, n_viz=N_VIZ, seed=0)

CLASS_NAME = cfg.get('class_name', '')
print(f'Built {len(replays)} greedy replays for visualisation.')


## 6 · Sample comparison — best / median / worst

In [ ]:
fig = plot_comparison(
    replays, baseline_cfg, cfg,
    class_idx=cfg.get('target_class', 1), class_name=CLASS_NAME,
    out_path=f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_comparison.png")
plt.show()


## 7 · Iteration playback — all steps on best-gain sample
Green contour = GT boundary. This is the UI Iteration Playback page data.

In [ ]:
# Playback the BEST-gain episode step-by-step
fig = plot_playback(
    replays[-1], cfg, class_name=CLASS_NAME,
    out_path=f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_playback.png")
plt.show()


## 8 · Behaviour analysis — trajectories + action distribution

In [ ]:
fig = plot_behaviour(
    replays, cfg, class_name=CLASS_NAME,
    out_path=f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_behaviour.png")
plt.show()


## 9 · Learning curves

In [ ]:
import matplotlib.pyplot as plt

# history is a DataFrame with columns: step, init_dice_mean, final_dice_mean,
# best_dice_mean, final_hd95_mean, delta_dice_mean
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['step'], history['init_dice_mean'],  label='Init Dice (U-Net)', ls='--', alpha=0.6, color='gray')
ax1.plot(history['step'], history['final_dice_mean'], label=f'Final Dice ({cfg["agent_type"]})', color='C0')
if 'best_dice_mean' in history.columns:
    ax1.plot(history['step'], history['best_dice_mean'], label='Best-seen Dice (ceiling)', color='C2', alpha=0.7)
ax1.set(xlabel='Training step', ylabel='Mean Val Dice',
        title=f"{cfg['dataset']} — {cfg['class_name']} refinement curve")
ax1.legend(); ax1.grid(alpha=0.3)

delta = history['final_dice_mean'] - history['init_dice_mean']
ax2.plot(history['step'], delta, color='C0')
ax2.axhline(0, color='k', lw=0.8)
ax2.fill_between(history['step'], delta, 0, where=(delta > 0), alpha=0.15, color='green')
ax2.fill_between(history['step'], delta, 0, where=(delta < 0), alpha=0.15, color='red')
ax2.set(xlabel='Training step', ylabel='Δ Dice (final − init)',
        title=f'Refinement gain — {cfg["agent_type"]} ({cfg["class_name"]})')
ax2.grid(alpha=0.3)

plt.suptitle(f"{cfg['dataset']} {cfg['agent_type']} — {cfg['class_name']} learning curves")
plt.tight_layout()
out = f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_curves.png"
plt.savefig(out, dpi=150); plt.show(); print(f'Saved to {out}')


## 10 · Test-set evaluation + summary JSON
Full test set (~7 min). Skip for dry-runs.

In [ ]:
import json
test_metrics = evaluate_testset(agent, test_samples, ENV_KW)
print(json.dumps(test_metrics, indent=2))

summary = {**test_metrics,
           'agent_type':    cfg['agent_type'],
           'target_class':  cfg['target_class'],
           'class_name':    cfg['class_name'],
           'dataset':       cfg['dataset'],
           'paradigm':      'local_mask_refinement'}
out = f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_test_results.json"
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved to', out)
print(f"\nBaseline (U-Net) Dice: {test_metrics['init_dice_mean']:.4f}")
print(f"Refined  (agent)  Dice: {test_metrics['final_dice_mean']:.4f}  "
      f"(Δ {test_metrics['delta_dice_mean']:+.4f})")
print(f"Best-seen ceiling Dice: {test_metrics['best_dice_mean']:.4f}")
